# Log-time decay analysis (block disorder, several control values at one L)

Reads the per-sample CSVs written by `../generators/get_slidding_p_time_log.jl` or
`../generators/get_upper_lower_binary_time_log.jl`: log-spaced output times, `blocklen` and
`time_log` in every file name, data under `stavskya_mc/data/time_log/`.

It runs the tests that separate a conventional power-law critical point from the
infinite-noise point of Barghathi, Vojta and Hoyos (BVH, arXiv:1603.08075). See
`REFEREE_CONFLICT_REVIEW.md`, sections 3 and 5.

| test | power-law critical point | BVH infinite-noise point |
|---|---|---|
| running $\delta_{\rm eff}(t)=\log_{10}[A(t/10)/A(t)]$ | plateau at $\delta$ | drifts to 0 like $\log_{10}[\ln t/\ln(t/10)]$ |
| $1/\delta_{\rm eff}$ vs $\ln t$ | flat | linear |
| $1/A$ vs $\ln t$ | curves upward | straight line |
| running $\alpha_{\rm eff}$ (log exponent) | grows without bound | $\to \bar\delta = 1$ |
| width of $P(-\ln A)$ over disorder | saturates | grows $\propto \ln t$ |
| crossing time $t_x(r)$ | $\ln t_x \approx \nu_t\ln(1/r)$ | $\ln t_x \propto r^{-1/2}$ |

$A = 1-\rho$ is the activity. All estimators live in `time_log_tools.py` and work on the log
grid directly, with no uniform `time_step` assumed. A vertical dotted line marks $t=L$: for
$t<L$ the disorder-averaged activity equals the infinite-chain value exactly (light-cone
argument, review 4.2.3).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))          # run from stavskya_mc/block_disorder/analysis
import time_log_tools as tl

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.2, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 13})
# plt.rcParams["text.usetex"] = True   # optional, needs a LaTeX install

In [ ]:
# ---- parameters (mirror the generator you ran) ------------------------------
MODEL = "slidding_p"            # "slidding_p" or "window_binary"
DATA_ROOT = "../../data/time_log"   # or e.g. "/Volumes/ExternalData/stavskya_mc/data/time_log"
L = 35000
BLOCK_LEN = 1
TIME_PREFACT = 100.0
PPD = 20                        # points_per_decade
N_SAMPLES = 4000
OFFSET = 0

# sliding-p model (get_slidding_p_time_log.jl)
UPPER_VAL = 0.43
LOWER_DIV = 20
P_C = 0.479
P_RATE = 0.0003
P_STEPS = list(range(-3, 4))

# upper/lower binary model (get_upper_lower_binary_time_log.jl)
AVG_EPS_C = 0.27033
AVG_EPS_RATE = 0.00005
P_VAL = 0.8
EPS_STEPS = list(range(-3, 4))

CONTROL_C = None                # current critical estimate; None -> middle control value
B = 10                          # time ratio for the running exponents
# fit window; None -> min(L, t_max)
FIT_TMIN = 1e2
FIT_TMAX = None
FIG_DIR = "figs"


In [ ]:
# Rebuild the parameter sets exactly like the Julia generators do
# (round(x, digits=6) in Julia <-> round(x, 6) here).
if MODEL == "slidding_p":
    MODEL_DIR = "time_rand_slidding_p"
    lower_val = round(UPPER_VAL / LOWER_DIV, 6)
    params = [dict(control=round(P_C + i * P_RATE, 6), u=UPPER_VAL, l=lower_val, p=round(P_C + i * P_RATE, 6))
              for i in P_STEPS]
    CONTROL_LABEL = r"$p$"
    INACTIVE_SIDE = +1          # larger p = more inactive steps
elif MODEL == "window_binary":
    MODEL_DIR = "time_rand_window_binary"
    f = P_VAL + (1 - P_VAL) / LOWER_DIV
    params = []
    for i in EPS_STEPS:
        u = round(AVG_EPS_C / f + i * (AVG_EPS_RATE / f), 6)
        l = round(u / LOWER_DIV, 6)
        params.append(dict(control=round(P_VAL * u + (1 - P_VAL) * l, 6), u=u, l=l, p=P_VAL))
    CONTROL_LABEL = r"$\bar\varepsilon$"
    INACTIVE_SIDE = +1          # larger mean healing = more inactive
else:
    raise ValueError(MODEL)

controls = [d["control"] for d in params]
if CONTROL_C is None:
    CONTROL_C = controls[len(controls) // 2]
Path(FIG_DIR).mkdir(exist_ok=True)
pd.DataFrame(params)

In [ ]:
runs = {}
for d in params:
    runs[d["control"]] = tl.load_time_log_run(DATA_ROOT, MODEL_DIR, L, d["u"], d["l"], d["p"], BLOCK_LEN,
                                              TIME_PREFACT, PPD, N_SAMPLES, offset=OFFSET)
t = runs[controls[0]].times
T_MAX = t[-1]
FIT_TMAX = FIT_TMAX or min(L, T_MAX)
summary = pd.DataFrame({c: {"samples": r.n, "t_max": r.times[-1], "surviving at t_max": r.surviving_fraction[-1],
                            "activity at t_max": r.mean[-1]} for c, r in runs.items()}).T
summary.index.name = "control"
summary

In [ ]:
cmap = plt.colormaps["viridis"].resampled(len(controls))
colors = {c: cmap(i) for i, c in enumerate(controls)}

def mark_light_cone(ax, x=np.log10):
    if L <= T_MAX:
        ax.axvline(x(L), color="k", ls=":", lw=1)

fig, ax = plt.subplots(figsize=(8, 5))
for c, r in runs.items():
    m = (r.times > 0) & (r.mean > 0)
    ax.plot(np.log10(r.times[m]), np.log10(r.mean[m]), color=colors[c], label=f"{c}")
mark_light_cone(ax)
ax.set_xlabel(r"$\log_{10} t$"); ax.set_ylabel(r"$\log_{10} A(t)$, $A = 1-\rho$")
ax.set_title(f"{MODEL}, L={L}, block_len={BLOCK_LEN}"); ax.legend(title=CONTROL_LABEL, fontsize=9)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_L{L}_bl{BLOCK_LEN}_activity.png", dpi=150)

## Running power-law exponent
A conventional critical point gives a plateau. BVH's $A\sim1/\ln t$ gives the dashed grey
curve, $\log_{10}[\ln t/\ln(t/10)]$, which keeps falling. Clean DP has $\delta = 0.159$.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for c, r in runs.items():
    T, d = tl.running_delta(r.times, r.mean, b=B)
    axs[0].plot(np.log10(T), d, color=colors[c], label=f"{c}")
    ok = d > 0.01                    # 1/delta_eff blows up where noise makes delta_eff ~ 0
    axs[1].plot(np.log(T[ok]), 1 / d[ok], color=colors[c])
Tg = np.logspace(np.log10(B * np.e), np.log10(T_MAX), 200)
axs[0].plot(np.log10(Tg), np.log10(np.log(Tg) / np.log(Tg / B)), "--", color="grey", label=r"$1/\ln t$")
axs[0].axhline(0.159, color="k", lw=0.8, label="clean DP")
mark_light_cone(axs[0]); mark_light_cone(axs[1], x=np.log)
axs[0].set_xlabel(r"$\log_{10} t$"); axs[0].set_ylabel(rf"$\delta_{{\rm eff}}=\log_{{{B}}}[A(t/{B})/A(t)]$")
axs[1].set_xlabel(r"$\ln t$"); axs[1].set_ylabel(r"$1/\delta_{\rm eff}$  (BVH: linear at criticality)"); axs[1].set_ylim(0, 40)
axs[0].legend(title=CONTROL_LABEL, fontsize=8)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_L{L}_bl{BLOCK_LEN}_delta_eff.png", dpi=150)

## BVH test: $1/A$ against $\ln t$
At the infinite-noise critical point $1/A = a + B\ln t$: a straight line with constant local
slope. Inactive-side curves bend up and active-side curves flatten.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for c, r in runs.items():
    m = (r.times > 0) & (r.mean > 0)
    axs[0].plot(np.log(r.times[m]), 1 / r.mean[m], color=colors[c], label=f"{c}")
    ts, s = tl.inverse_activity_slope(r.times, r.mean, span=2.0)
    axs[1].plot(np.log(ts), s, color=colors[c])
mark_light_cone(axs[0], x=np.log); mark_light_cone(axs[1], x=np.log)
axs[0].set_xlabel(r"$\ln t$"); axs[0].set_ylabel(r"$1/A$")
axs[1].set_xlabel(r"$\ln t$"); axs[1].set_ylabel(r"$d(1/A)/d\ln t$ (over a factor 2 in $t$)")
axs[0].legend(title=CONTROL_LABEL, fontsize=8)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_L{L}_bl{BLOCK_LEN}_inverse_activity.png", dpi=150)

## Running log-exponent
$\alpha_{\rm eff}(t)=\ln[A(t/b)/A(t)]/\ln[\ln t/\ln(t/b)]$ is constant for $A\propto(\ln t)^{-\alpha}$,
and BVH predict $\alpha\to\bar\delta=1$. This replaces the old $1/\ln\ln t\to0$ extrapolation,
which needed a 5–10× lever arm (review 4.3.4).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for c, r in runs.items():
    T, a = tl.running_alpha(r.times, r.mean, b=B)
    ax.plot(np.log10(T), a, color=colors[c], label=f"{c}")
ax.axhline(1.0, color="k", lw=0.8, label=r"BVH $\bar\delta=1$")
mark_light_cone(ax)
ax.set_xlabel(r"$\log_{10} t$"); ax.set_ylabel(r"$\alpha_{\rm eff}$"); ax.legend(title=CONTROL_LABEL, fontsize=8)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_L{L}_bl{BLOCK_LEN}_alpha_eff.png", dpi=150)

## Fit comparison over one window
Weighted fits of three forms over `[FIT_TMIN, FIT_TMAX]`. The $\chi^2_\nu$ values ignore the
strong correlation between times (the same trajectories feed every point), so compare them
with each other but do not read them as absolute goodness of fit.

In [ ]:
rows = []
for c, r in runs.items():
    fp = tl.fit_power_law(r.times, r.mean, r.sem, FIT_TMIN, FIT_TMAX)
    fb = tl.fit_bvh(r.times, r.mean, r.sem, FIT_TMIN, FIT_TMAX)
    fl = tl.fit_log_power(r.times, r.mean, r.sem, FIT_TMIN, FIT_TMAX)
    rows.append({"control": c, "power: delta": fp["delta"], "power: chi2r": fp["chi2r"],
                 "BVH 1/A=a+B ln t: B": fb["B"], "BVH: ln t0=a/B": fb["ln_t0"], "BVH: chi2r": fb["chi2r"],
                 "(ln t)^-alpha: alpha": fl["alpha"], "log-power: chi2r": fl["chi2r"], "points": fp["n"]})
fits = pd.DataFrame(rows).set_index("control")
print(f"fit window: {FIT_TMIN:g} <= t <= {FIT_TMAX:g}")
fits.round(4)

## Width of the distribution over disorder realizations
Each sample has its own disorder sequence, and the spatial average over $L$ sites removes
most of the intrinsic noise. So the spread of $x_i=-\ln A_i(t)$ across samples measures the
disorder broadening. BVH predict ${\rm std}(x)\propto\ln t$ at criticality; a
finite-disorder fixed point saturates. Only surviving samples are counted, and the right
panel shows what fraction that is.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
width_slopes = {}
for c, r in runs.items():
    ts, sd, surv = tl.log_activity_width(r)
    m = (ts > 0) & np.isfinite(sd)
    axs[0].plot(np.log(ts[m]), sd[m], color=colors[c], label=f"{c}")
    axs[1].plot(np.log10(ts[ts > 0]), surv[ts > 0], color=colors[c])
    w = m & (ts >= FIT_TMIN) & (ts <= FIT_TMAX)
    if w.sum() >= 3:
        width_slopes[c] = np.polyfit(np.log(ts[w]), sd[w], 1)[0]
mark_light_cone(axs[0], x=np.log); mark_light_cone(axs[1])
axs[0].set_xlabel(r"$\ln t$"); axs[0].set_ylabel(r"std over samples of $-\ln A_i$")
axs[1].set_xlabel(r"$\log_{10} t$"); axs[1].set_ylabel("surviving fraction")
axs[0].legend(title=CONTROL_LABEL, fontsize=8)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_L{L}_bl{BLOCK_LEN}_log_width.png", dpi=150)
pd.Series(width_slopes, name="d std(x) / d ln t over fit window").round(4)

## Crossing times off criticality
For every control value on the inactive side of `CONTROL_C`, $t_x$ is the first time
$1/A$ exceeds 1.1 × the critical curve's value (as in BVH Fig. 7). Compare the two
linearizations:
* BVH: $r^{-1/2}$ linear in $\ln t_x$;
* power law: $\ln t_x$ linear in $\ln(1/r)$, slope $\nu_t$.

With only a few control values this is a first look; add more values on the inactive side
to make it quantitative.

In [ ]:
A_c = runs[CONTROL_C].mean
xs = []
for c, r in runs.items():
    rr = INACTIVE_SIDE * (c - CONTROL_C) / CONTROL_C
    if rr <= 0:
        continue
    tx = tl.crossing_time(t, A_c, r.mean, factor=1.1)
    xs.append({"control": c, "r": rr, "t_x": tx})
cross = pd.DataFrame(xs)
display(cross)
ok = cross.dropna() if len(cross) else cross
if len(ok) >= 2:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
    axs[0].plot(np.log(ok.t_x), ok.r ** -0.5, "o-"); axs[0].set_xlabel(r"$\ln t_x$"); axs[0].set_ylabel(r"$r^{-1/2}$ (BVH: linear)")
    axs[1].plot(np.log(1 / ok.r), np.log(ok.t_x), "o-"); axs[1].set_xlabel(r"$\ln(1/r)$"); axs[1].set_ylabel(r"$\ln t_x$ (power law: slope $\nu_t$)")
    fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_L{L}_bl{BLOCK_LEN}_crossing.png", dpi=150)
    print("power-law slope nu_t(eff) =", np.polyfit(np.log(1 / ok.r), np.log(ok.t_x), 1)[0])
else:
    print("fewer than two inactive-side control values crossed the critical curve; nothing to fit")